# Module 315 — Vulnerable MCP Servers

This companion notebook turns Section 11 into a controlled, annotated client-side review. It demonstrates capability enumeration and the request shapes used by the HTB lesson for error disclosure, SQL injection, command injection, and SSRF.

**Scope and safe defaults:** replace the target placeholder only with the IP and port of the currently spawned HTB lab. Every network probe is disabled by default. The lab may return sensitive values or trigger server-side actions; inspect one probe at a time, keep outputs out of Git, and stop when the HTB section is complete. This notebook is for the authorized Academy target, not a general scanner.


## 1. Configure the current lab target

The endpoint below follows the MCP HTTP path used in the lesson. Keep host and port in variables so a refreshed HTB instance can be configured without editing request code.

RUN_LAB_REQUESTS is a deliberate safety switch. Merely running the setup cell does not connect to the target. Set it to True only after entering the current spawned target and while working inside that HTB section.


In [ ]:
# Uncomment in the AI environment if FastMCP is not installed:
# %pip install fastmcp

import re
from fastmcp import Client

TARGET_HOST = "REPLACE_WITH_CURRENT_HTB_TARGET_IP"
TARGET_PORT = 8000
MCP_PATH = "/mcp/"

# False means notebook cells will describe what they would do but send no requests.
RUN_LAB_REQUESTS = False

def require_lab_enabled():
    if not RUN_LAB_REQUESTS:
        raise RuntimeError("Lab requests are disabled. Set RUN_LAB_REQUESTS=True for the current HTB target.")
    if TARGET_HOST == "REPLACE_WITH_CURRENT_HTB_TARGET_IP":
        raise ValueError("Set TARGET_HOST to the current spawned HTB target IP first.")

MCP_URL = f"http://{TARGET_HOST}:{TARGET_PORT}{MCP_PATH}"
client = Client(MCP_URL)

print("Configured endpoint:", MCP_URL)
print("Requests enabled:", RUN_LAB_REQUESTS)


## 2. Enumerate the exposed capabilities

The three list calls ask what the server exposes; they do not call each capability. Read the results as an attack-surface inventory:

- A resource is a readable data endpoint.
- A resource template has caller-supplied URI parameters, shown in braces.
- A tool is an operation the client can invoke and may have side effects.

Descriptions help you choose what to inspect, but they are not security checks. A capability can be described as safe and still have an unsafe implementation.


In [ ]:
async def enumerate_capabilities():
    require_lab_enabled()
    async with client:
        resources = await client.list_resources()
        templates = await client.list_resource_templates()
        tools = await client.list_tools()

    print("Resources:")
    for resource in resources:
        print(f"  {resource.name}: {(resource.description or '').strip()}")

    print("\nResource templates:")
    for template in templates:
        print(f"  {template.uriTemplate}: {(template.description or '').strip()}")

    print("\nTools:")
    for tool in tools:
        properties = tool.inputSchema.get("properties", {})
        print(f"  {tool.name}({', '.join(properties)}): {(tool.description or '').strip()}")

if RUN_LAB_REQUESTS:
    await enumerate_capabilities()
else:
    print("Enumeration skipped. Enable requests only for the current HTB lab.")


## 3. Probe an error path for information disclosure

HTB's invalid-item example asks the quantity API for a name that should not exist. The goal is to inspect whether the client-facing error exposes upstream request details, headers, credentials, or internal paths.

The helper below redacts common API-key fields before printing an exception. Redaction is a precaution, not proof that all secrets are removed. If the response contains anything sensitive, do not save the raw output or commit it.


In [ ]:
def redact_sensitive_text(text: str) -> str:
    # Mask common key/token values if the server places them in an exception.
    patterns = [
        r"(?i)(x-api-key\s*['\"]?\s*[:=]\s*['\"]?)[^'\",}\s]+",
        r"(?i)((?:api[_ -]?key|token|secret)\s*[:=]\s*['\"]?)[^'\",}\s]+",
    ]
    for pattern in patterns:
        text = re.sub(pattern, r"\1[REDACTED]", text)
    return text

async def probe_invalid_quantity(item="not-a-real-item"):
    require_lab_enabled()
    async with client:
        try:
            result = await client.read_resource(f"quantity://{item}")
            print("Response:", [getattr(part, "text", str(part)) for part in result])
        except Exception as exc:
            print("Redacted error:", redact_sensitive_text(str(exc)))

if RUN_LAB_REQUESTS:
    await probe_invalid_quantity()
else:
    print("Error-path probe skipped.")


## 4. Recognize the SQL injection sequence

The HTB lesson builds from a normal item lookup, to a quote that may break a quoted SQL value, then to a comment suffix, and finally to a UNION SELECT example. %20 is URL encoding for a space. It can pass a URL validator that rejects literal spaces and then become a space after a later decoding step. This illustrates why validation must match the parser that ultimately consumes the value.

The following helper sends only a resource read to the current HTB target. Run individual cases, observe the result, then stop. The final case is an educational SQL-injection probe and may return data from the lab database.


In [ ]:
async def read_price_resource(uri: str):
    require_lab_enabled()
    async with client:
        try:
            result = await client.read_resource(uri)
            return [getattr(part, "text", str(part)) for part in result]
        except Exception as exc:
            return [redact_sensitive_text(str(exc))]

# HTB sequence, as strings for deliberate one-at-a-time use:
SQLI_CASES = {
    "baseline": "price://banana",
    "quote_probe": "price://banana'",
    "comment_probe": "price://banana'--",
    "encoded_union_demo": "price://x'%20UNION%20SELECT%201--",
}

# Example only; to send one case, explicitly call:
# await read_price_resource(SQLI_CASES["baseline"])
#
# The notebook does not submit probes automatically.


## 5. Recognize command injection

A command separator is shell syntax that can end one command and start another. HTB demonstrates a semicolon in the tool argument. If the server builds a shell command by concatenating untrusted text, the shell may interpret the added text as another command.

This is a lab-only request. It may execute an operating-system command on the target. Do not call it on any non-HTB host.


In [ ]:
async def call_command_tool(command: str):
    require_lab_enabled()
    async with client:
        try:
            result = await client.call_tool(
                "execute_server_command",
                {"command": command},
            )
            return [getattr(part, "text", str(part)) for part in result.content]
        except Exception as exc:
            return [redact_sensitive_text(str(exc))]

COMMAND_CASES = {
    "allowed_baseline": "date",
    "separator_demo": "date;id",  # HTB's intentionally vulnerable lab example
}

# Explicitly send one only while the current HTB target is active:
# await call_command_tool(COMMAND_CASES["allowed_baseline"])
# await call_command_tool(COMMAND_CASES["separator_demo"])


## 6. Recognize SSRF

With server-side request forgery (SSRF), the server—not your notebook—makes an HTTP request to a URL you supply. This can let a caller reach destinations available from the server's network. A success/refused difference for loopback ports can disclose which internal services are reachable.

Only use these destinations against the current spawned HTB lab. A callback URL should point to a listener you control inside the authorized lab environment; never use a third party's host.


In [ ]:
async def call_fetch_tool(url: str):
    require_lab_enabled()
    async with client:
        try:
            result = await client.call_tool(
                "fetch_price_data",
                {"url": url},
            )
            return [getattr(part, "text", str(part)) for part in result.content]
        except Exception as exc:
            return [redact_sensitive_text(str(exc))]

SSRF_LAB_CASES = {
    "loopback_port_80": "http://127.0.0.1:80",
    "loopback_port_22": "http://127.0.0.1:22",
    # Replace with a listener you control and can reach from the lab, if required:
    "controlled_callback": "http://YOUR_AUTHORIZED_LISTENER:8000/ssrf",
}

# Explicit, one-at-a-time lab calls:
# await call_fetch_tool(SSRF_LAB_CASES["loopback_port_80"])


## 7. Findings-to-controls study table

| Evidence observed | What it suggests | Defensive boundary |
| --- | --- | --- |
| Exception includes a key, request header, or path | Error-based disclosure | Redacted server logs; generic client errors; rotate exposed credentials |
| Changing an object ID returns another user's record | IDOR / missing object authorization | Authorize the current user for that exact object on each request |
| Quote/comment/UNION changes a price result | SQL injection | Parameterized query and allowlisted item identifier |
| A separator causes an extra OS command | Shell command injection | Avoid shell; fixed executable and argument list |
| Caller-selected URL reaches loopback/private hosts | SSRF | Destination allowlist, normalized URL and IP validation, DNS/redirect checks, egress restrictions |

For each finding, write down: untrusted input → capability → downstream interpreter/service → missing control → impact. This chain links an observed clue to both the vulnerability name and the fix.


## COAE quick recall

- MCP is a protocol, not an authorization system. A direct network client can call exposed capabilities without an LLM.
- Enumerate resources, resource templates, and tools; then trace each argument to the code or service that consumes it.
- Error messages and logs can leak secrets even when the normal response does not.
- IDOR means an object identifier is accepted without checking that the current caller may access that object.
- Parameterized SQL prevents input from becoming SQL syntax; URI validation alone does not.
- Shell separators matter only when untrusted input reaches a shell command parser.
- SSRF means the server fetches a caller-influenced destination; use destination validation and outbound network controls.
- Validation asks whether input has an acceptable shape/value; authorization asks whether this caller may perform this action on this object.
- HTB lab probes here are intentionally disabled by default. A code cell being present does not mean it was executed.
